<a href="https://colab.research.google.com/github/salsilsulselsol/Associate-Data-Scientist-Python-Nasional-Notes-Experiment/blob/main/10_data_cleansing_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 10: Data Cleansing for Banking Operations
**Dataset:** Banking Dirty Dataset *https://github.com/john-e-burris82/cleaning_banking_data/blob/main/banking_dirty.csv

**Objective:** Resolve structural, logical, and quality issues to prepare data for financial analysis.

### Cleansing Roadmap:
1. **Structural Audit**: Identify missing values, data types, and duplicates.
2. **Logical Validation**: Correct temporal anomalies (e.g., transaction dates vs. account opening).
3. **Imputation & Correction**: Handle missing values and age-DOB inconsistencies.
4. **Standardization**: Normalize categorical strings and numeric scales.

## Section 1: Data Loading and Initial Audit

In [1]:
import pandas as pd
import numpy as np

# Load the dirty dataset (Adjust filename if necessary)
file_name = 'banking_dirty.csv'
try:
    df = pd.read_csv(file_name)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: {file_name} not found. Please upload the file to Colab.")

# 1.1 Structural Inspection
print("\n--- Data Schema ---")
print(df.info())

# 1.2 Quality Audit: Missing Values & Duplicates
print("\n--- Data Quality Report ---")
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"\nTotal Duplicate Rows: {df.duplicated().sum()}")

Dataset loaded successfully.

--- Data Schema ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        100 non-null    int64  
 1   cust_id           100 non-null    object 
 2   birth_date        100 non-null    object 
 3   Age               100 non-null    int64  
 4   acct_amount       100 non-null    float64
 5   inv_amount        100 non-null    int64  
 6   fund_A            100 non-null    float64
 7   fund_B            100 non-null    float64
 8   fund_C            100 non-null    float64
 9   fund_D            100 non-null    float64
 10  account_opened    100 non-null    object 
 11  last_transaction  100 non-null    object 
dtypes: float64(5), int64(3), object(4)
memory usage: 9.5+ KB
None

--- Data Quality Report ---
Missing Values:
Unnamed: 0          0
cust_id             0
birth_date          0
Age       

## Section 2: Fixing Dates and Logical Mismatches

In [2]:
import pandas as pd

# 2.1 Robust Date Conversion with Mixed Format Handling
# Using format='mixed' to suppress UserWarnings and handle multiple date formats
date_cols = ['birth_date', 'account_opened', 'last_transaction']

for col in date_cols:
    if col in df.columns:
        # Remove white spaces to handle unreadable characters (as per data error documentation)
        df[col] = df[col].astype(str).str.strip()

        # Explicitly parse dates using mixed format to ensure consistency
        df[col] = pd.to_datetime(df[col], dayfirst=True, format='mixed', errors='coerce')

# 2.2 Logical Check: Transaction after Account Opening
# Identifying impossible values where last_transaction occurs before account_opened
if 'account_opened' in df.columns and 'last_transaction' in df.columns:
    inconsistent_mask = df['last_transaction'] < df['account_opened']
    inconsistent_count = inconsistent_mask.sum()
    print(f"Inconsistent Transaction Dates Found: {inconsistent_count}")

    # Correction: Aligning logic by replacing invalid dates with account_opening date
    df.loc[inconsistent_mask, 'last_transaction'] = df['account_opened']

# 2.3 Logical Check: Age vs. Birth Date Consistency
current_year = 2026
if 'age' in df.columns and 'birth_date' in df.columns:
    # Calculating the correct age based on the birth_date attribute
    df['calculated_age'] = current_year - df['birth_date'].dt.year

    # Identify and correct mismatches to ensure data integrity
    age_mismatch = df[df['age'] != df['calculated_age']]
    if not age_mismatch.empty:
        print(f"Age Mismatches Corrected: {len(age_mismatch)}")
        df['age'] = df['calculated_age']

    # Drop the temporary calculation column
    df.drop(columns=['calculated_age'], inplace=True)

print("\n--- Temporal Validation Complete ---")
print(df[date_cols].dtypes)

Inconsistent Transaction Dates Found: 12

--- Temporal Validation Complete ---
birth_date          datetime64[ns]
account_opened      datetime64[ns]
last_transaction    datetime64[ns]
dtype: object


## Section 3: Imputation and Outlier Management

In [3]:
import pandas as pd
import numpy as np

# 3.1 Identifying Missing Values
# Following the data preprocessing task to identify missing values
print("--- Missing Values Before Imputation ---")
print(df.isnull().sum())

# 3.2 Imputing Categorical Data (Gender)
# Strategy: Use Mode for categorical attributes
if 'gender' in df.columns:
    # Filling missing values with the most frequent value
    gender_mode = df['gender'].mode()[0]
    df['gender'] = df['gender'].fillna(gender_mode)

# 3.3 Imputing Numerical Data (Income)
# Strategy: Use Median to avoid bias from potential outliers
if 'income' in df.columns:
    income_median = df['income'].median()
    df['income'] = df['income'].fillna(income_median)

# 3.4 Outlier Identification and Correction (Income)
# Identifying and treating outliers is a core task in data cleansing
if 'income' in df.columns:
    # Calculating IQR boundaries
    Q1 = df['income'].quantile(0.25)
    Q3 = df['income'].quantile(0.75)
    IQR = Q3 - Q1

    # Defining outlier thresholds
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Count outliers before correction
    outliers = df[(df['income'] < lower_limit) | (df['income'] > upper_limit)]
    print(f"\nOutliers detected in Income: {len(outliers)}")

    # Applying Clipping method to handle outliers
    # This keeps the data volume consistent while removing extreme variance
    df['income'] = df['income'].clip(lower_limit, upper_limit)

print("\nSection 3: Imputation and Outlier handling completed.")

--- Missing Values Before Imputation ---
Unnamed: 0          0
cust_id             0
birth_date          0
Age                 0
acct_amount         0
inv_amount          0
fund_A              0
fund_B              0
fund_C              0
fund_D              0
account_opened      0
last_transaction    0
dtype: int64

Section 3: Imputation and Outlier handling completed.


## Section 4: Standardization and Final Validation

In [4]:
# 4.1 Categorical String Standardization
# Fixing inconsistent values by removing white spaces and normalizing case
categorical_features = df.select_dtypes(include=['object']).columns
for col in categorical_features:
    # Strip leading/trailing whitespaces and convert to Title Case
    df[col] = df[col].astype(str).str.strip().str.title()

# 4.2 Removing Duplicate Records
# Duplicates can lead to biased analysis and must be removed
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Duplicate rows removed: {initial_rows - df.shape[0]}")

# 4.3 Final Quality Audit
# Verifying that the data is now clean and ready for analysis
print("\n--- Final Clean Data Audit ---")
print(df.isnull().sum())
print(f"\nFinal Dataset Dimensions: {df.shape}")

# 4.4 Exporting the Cleaned Dataset
# Saving the output for the Modeling phase
output_file = 'banking_clean_data.csv'
df.to_csv(output_file, index=False)
print(f"\nCleaned dataset exported to: {output_file}")

Duplicate rows removed: 0

--- Final Clean Data Audit ---
Unnamed: 0          0
cust_id             0
birth_date          0
Age                 0
acct_amount         0
inv_amount          0
fund_A              0
fund_B              0
fund_C              0
fund_D              0
account_opened      0
last_transaction    0
dtype: int64

Final Dataset Dimensions: (100, 12)

Cleaned dataset exported to: banking_clean_data.csv
